# Homework 3: Sparse Autoencoders for Image Interpretability

In this homework, you will use a **sparse autoencoder (SAE)** to analyze frozen image embeddings from a pretrained CLIP model on a small **binary bird classification** dataset of puffins and penguins.



## What you will build

You will complete four core components:

1. **A linear probe** on frozen CLIP features  
2. **An SAE training loop** to learn a sparse dictionary over those features  
3. **A top-activating-example retrieval function** for interpreting latents  
4. **A latent ablation experiment** that tests whether a latent matters for classification

---

![Pipeline](https://raw.githubusercontent.com/aclarkse/eai_hw3/main/pipeline.png)

---

### Part 1: Feature extraction with CLIP

We use [CLIP](https://openai.com/research/clip) (Contrastive Language–Image Pre-Training) as a frozen feature extractor. CLIP's Vision Transformer (ViT) processes each image and produces a **512-dimensional embedding** that encodes high-level visual semantics. Because CLIP is trained to align images with text descriptions, its embedding space tends to organise around concepts you can describe in words, things like "orange beak" or "icy background".

This is directly analogous to how SAE interpretability is applied to language models: instead of analysing a residual stream, we analyse CLIP's image embeddings. You do not need to train CLIP — it is frozen throughout this homework.

---

### Part 2: Baseline linear probe

Before training any SAE, we need to establish a baseline by fitting a **logistic regression** directly on the frozen CLIP features. This tells us:
- how separable the two classes already are in raw CLIP space
- what accuracy we should expect downstream when we probe SAE latents

A linear probe is intentionally simple: if it achieves high accuracy, it confirms that the class-relevant information is linearly accessible in the embeddings. The SAE's job is to tell us *which directions* encode that information.

---

### Part 3: Training the SAE

You will then train a **Sparse Autoencoder** (SAE) on top of the frozen CLIP features. The SAE learns a sparse overcomplete basis, a much larger set of directions (4096 vs 512) that attempts to disentangle features that are packed together in each CLIP embedding.

The SAE is trained with two competing objectives:
- **Reconstruction loss**: the decoded output should match the original CLIP embedding
- **L1 sparsity penalty**: only a small number of latents should be active for any given image

You will need to track the following two diagnostics during training:
- **Reconstruction loss** — how faithfully the SAE reproduces the input (lower is better)
- **Dead latent %** — the fraction of latents that never activate; this rises as the L1 penalty prunes weak latents then plateaus, which is expected behaviour

---

### Part 4: Latent retrieval & naming

Once the SAE is trained, you will analyse what its latents have learned in two ways:

**Top-activating image retrieval**: for a given latent, find the images that activate it most strongly. If a latent is interpretable, these images should share a common visual concept (e.g. all showing a colourful beak, or all showing a snowy background).

**Latent naming with CLIP text**: once we have found class-specific latents and their top-activating images, we can automatically label what each latent detects by comparing it to a bank of text descriptions. The best-matching descriptions tell us what visual concept the latent has learned to detect, and we can verify this by retrieving the training image closest to each description.

## How to work through the notebook


- Cells with `TODO` blocks are the places where you must write code.
- Do **not** change function names, input arguments, or return types. The autograder expect those exactly.

## What you need before running this notebook

You have been provided with a file called `birds.zip`.

**Before running any cells, upload `birds.zip` to your Colab session and run the setup cells: the notebook will extract it automatically.** You do not need to unzip it yourself.

After extraction, the notebook expects this folder structure:
```
birds/
├── train/
│   ├── puffin/
│   │   ├── image001.jpg
│   │   └── ...
│   └── penguin/
│       ├── image001.jpg
│       └── ...
└── test/
    ├── puffin/
    └── penguin/
```

If you see a `FileNotFoundError`, check that `birds.zip` is in the same directory as the notebook (in Colab: visible in the left-hand file browser panel).

## Submission Requirements

Export your completed notebook as a PDF that clearly displays all code, outputs, and visualizations. Verify the PDF renders properly with readable font sizes and no truncated content. Include all discussion responses and ensure mathematical notation displays correctly. Submit the PDF and the Python Notebook (`.ipynb`) on Gradescope before the specified deadline.

# Setup
Run the next two code cells first. The install cell may take a minute the first time.


In [ ]:
%pip -q install transformers datasets torch scikit-learn rich jaxtyping tqdm pillow

In [ ]:
import os
import random
import zipfile
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch as t
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from torch import Tensor
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
t.manual_seed(SEED)

device = t.device("cuda" if t.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Load the image dataset

The next cell searches for the dataset in your runtime. If it raises an error, the message tells you exactly what file or folder is missing.


In [ ]:
def locate_bird_dataset() -> Path:
    """Return the path to a local birds/ directory, extracting birds.zip if needed."""
    folder_candidates = [
        Path("birds"),
        Path("data/birds"),
        Path("/mnt/data/birds"),
        Path("/content/birds"),
    ]
    for folder in folder_candidates:
        if folder.exists() and folder.is_dir():
            return folder

    zip_candidates = [
        Path("birds.zip"),
        Path("data/birds.zip"),
        Path("/mnt/data/birds.zip"),
        Path("/content/birds.zip"),
    ]
    extract_root = Path("data")
    extract_root.mkdir(exist_ok=True)
    extracted_dir = extract_root / "birds"

    for zip_path in zip_candidates:
        if zip_path.exists():
            if not extracted_dir.exists():
                print(f"Extracting {zip_path} to {extracted_dir} ...")
                with zipfile.ZipFile(zip_path, "r") as zf:
                    zf.extractall(extract_root)
            if extracted_dir.exists():
                return extracted_dir

    raise FileNotFoundError(
        "Could not find birds/ or birds.zip. Put one of them in the notebook directory "
        "(or in data/), then rerun this cell."
    )

bird_dir = locate_bird_dataset()
print(f"Loading dataset from: {bird_dir}")

full_dataset = load_dataset("imagefolder", data_dir=str(bird_dir), split="train")
full_dataset = full_dataset.rename_column("label", "binary_label")

CLASS_NAMES = list(full_dataset.features["binary_label"].names)
assert len(CLASS_NAMES) == 2, f"Expected exactly 2 classes, found {len(CLASS_NAMES)}: {CLASS_NAMES}"

split = full_dataset.train_test_split(test_size=0.25, seed=SEED)
train_ds = split["train"]
test_ds = split["test"]

print(f"Classes: {CLASS_NAMES}")
print(f"Train size: {len(train_ds)}")
print(f"Test size:  {len(test_ds)}")


In [ ]:
def show_sample_images(dataset, n_per_class: int = 4) -> None:

    fig, axes = plt.subplots(len(CLASS_NAMES), n_per_class, figsize=(3 * n_per_class, 3 * len(CLASS_NAMES)))
    if len(CLASS_NAMES) == 1:
        axes = np.array([axes])
    if n_per_class == 1:
        axes = axes[:, None]

    for class_idx, class_name in enumerate(CLASS_NAMES):
        class_examples = [ex for ex in dataset if ex["binary_label"] == class_idx][:n_per_class]
        for j, ex in enumerate(class_examples):
            axes[class_idx, j].imshow(ex["image"].convert("RGB"))
            axes[class_idx, j].set_title(f"{class_name}")
            axes[class_idx, j].axis("off")
    plt.tight_layout()
    plt.show()

show_sample_images(train_ds, n_per_class=3)


# Part 1: CLIP feature extraction

## How it works

CLIP (Contrastive Language–Image Pre-Training) is a model trained by OpenAI on hundreds of millions of (image, text) pairs scraped from the internet. The key idea is **contrastive training**: given a batch of image-text pairs, CLIP learns to map matching pairs close together in a shared embedding space, while pushing non-matching pairs apart.

It has two components trained jointly:
- A **vision encoder** (a Vision Transformer, ViT) that maps images to vectors
- A **text encoder** (a Transformer) that maps text descriptions to vectors

After training, an image of a puffin and the text "a puffin with an orange beak" will be nearby in this shared space, while "a penguin on ice" will be far away.

## How we use it

We use CLIP purely as a **frozen feature extractor**: we will load the pretrained weights and never update them. For each image, we:

1. Pass it through the ViT vision encoder
2. Take the output of the `[CLS]` token — a special token prepended to the image patches whose final hidden state aggregates information from the whole image
3. Project it through a linear layer to get a **512-dimensional embedding**
4. L2-normalise the result so all embeddings lie on the unit sphere

The result is a single 512-dim vector per image that encodes its high-level visual content. This is what we feed into the linear probe and the SAE to study their internal stucture, with neither of these models ever needing to see the raw pixels.




In [ ]:
print("Loading CLIP image encoder...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval();

D_CLIP = clip_model.config.projection_dim
print("CLIP feature dimension:", D_CLIP)

In [ ]:
@t.no_grad()
def extract_clip_features(dataset, batch_size: int = 64, desc: str = "Extracting"):
    """
    Return:
        feats:  Tensor of shape (n_examples, D_CLIP), L2-normalized
        labels: Tensor of shape (n_examples,)
    """
    all_feats = []
    all_labels = []

    for start in tqdm(range(0, len(dataset), batch_size), desc=desc):
        batch = dataset[start : start + batch_size]
        images = [img.convert("RGB") for img in batch["image"]]
        labels = batch["binary_label"]

        inputs = clip_processor(images=images, return_tensors="pt").to(device)
        outputs = clip_model.vision_model(pixel_values=inputs["pixel_values"])
        feats = clip_model.visual_projection(outputs.pooler_output)
        feats = F.normalize(feats, dim=-1)

        all_feats.append(feats.cpu())
        all_labels.append(t.tensor(labels, dtype=t.long))

    return t.cat(all_feats, dim=0), t.cat(all_labels, dim=0)

train_feats, train_labels = extract_clip_features(train_ds, desc="Train features")
test_feats, test_labels = extract_clip_features(test_ds, desc="Test features")

print("Train feature shape:", tuple(train_feats.shape))
print("Test feature shape: ", tuple(test_feats.shape))


In [ ]:
# PCA visualization of CLIP embeddings
pca = PCA(n_components=2)
train_2d = pca.fit_transform(train_feats.numpy())

plt.figure(figsize=(7, 5))
labels_np = train_labels.numpy()

for i, class_name in enumerate(CLASS_NAMES):
    idx = labels_np == i
    plt.scatter(
        train_2d[idx, 0],
        train_2d[idx, 1],
        label=class_name,
        alpha=0.65
    )

plt.title("PCA of frozen CLIP image embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Class")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Variance explained by PC1 + PC2: {pca.explained_variance_ratio_[:2].sum():.3f}")

The PCA plot gives a coarse geometric view of the features, but it does **not** tell us which internal directions or latent features matter.  
That is the job of the linear probe and the SAE.

# Part 2: Baseline linear probe

Next we will fit a **linear probe**, a simple classifier trained on top of frozen features we got from CLIP. If it performs well, that would suggest that class information is already linearly accessible in CLIP space.

## Your task

Implement `fit_linear_probe` with the following behavior:

- train a `LogisticRegression` classifier on `train_feats` and `train_labels`
- compute predictions on the test set
- return both:
  1. the fitted sklearn model
  2. a metrics dictionary with keys:
     - `"accuracy"`
     - `"n_test"`
     - `"coef_shape"`

### Required function signature

```python
def fit_linear_probe(
    train_feats: np.ndarray,
    train_labels: np.ndarray,
    test_feats: np.ndarray,
    test_labels: np.ndarray,
) -> tuple[LogisticRegression, dict]:
```

### Hints

- Use `max_iter=1000` and `random_state=SEED`
- `accuracy_score` is already imported
- `probe.coef_.shape` should be a tuple like `(1, D_CLIP)` for binary logistic regression



### Implementation task: `fit_linear_probe`

You will now implement a baseline classifier on top of frozen CLIP features.

**What you should do**
1. Create a `LogisticRegression` model from scikit-learn.
2. Fit it on the training features and labels.
3. Predict on the test features.
4. Return both:
   - the fitted model object
   - a dictionary with a few summary statistics

**Hints**
- A *linear probe* is just a simple classifier trained on top of fixed features.
- In a basic ML course, logistic regression is a standard choice for binary classification.
- `accuracy_score(y_true, y_pred)` returns a scalar accuracy.
- `probe.coef_.shape` should look like `(1, d)` for binary classification with `d` input features.


In [ ]:
def fit_linear_probe(
    train_feats: np.ndarray,
    train_labels: np.ndarray,
    test_feats: np.ndarray,
    test_labels: np.ndarray,
) -> tuple[LogisticRegression, dict]:
    """Fit logistic regression on frozen CLIP features."""
    # TODO: create a logistic regression classifier.
    # Hint: use max_iter=1000 and random_state=SEED so results are stable.
    probe = LogisticRegression(max_iter=1000, random_state=SEED)

    # TODO: train the classifier on the training features
    probe.fit(train_feats, train_labels)

    # TODO: compute predicted labels on the test set.
    preds = probe.predict(test_feats)

    # TODO: build the metrics dictionary with keys:
    #   "accuracy"  -> float test accuracy
    #   "n_test"    -> number of test examples as an int
    #   "coef_shape"-> shape of probe.coef_ as a tuple
    metrics = {"accuracy": accuracy_score(test_labels, preds), "n_test": int(len(test_labels)), "coef_shape": probe.coef_.shape}

    return probe, metrics

In [ ]:
# Fit linear probe and evaluate
probe, probe_metrics = fit_linear_probe(
    train_feats.numpy(),
    train_labels.numpy(),
    test_feats.numpy(),
    test_labels.numpy(),
)
print("Probe metrics:", probe_metrics)
print(classification_report(test_labels.numpy(), probe.predict(test_feats.numpy()), target_names=CLASS_NAMES))


# Sparse autoencoder (SAE) background

We now train a sparse autoencoder on the CLIP feature vectors.

For an input vector $x \in \mathbb{R}^{d_{in}}$, the SAE computes:

$$
z = \mathrm{ReLU}((x - b_{dec})W_{enc} + b_{enc})
$$

$$
\hat{x} = zW_{dec} + b_{dec}
$$

and minimizes

$$
\mathcal{L}(x, z, \hat{x}) = \underbrace{\|x - \hat{x}\|_2^2}_{\text{reconstruction}} + \lambda \underbrace{\|z\|_1}_{\text{sparsity}}.
$$

## Mental model

- The encoder proposes many candidate latent features.
- ReLU keeps only positive activations.
- The L1 penalty encourages **most latents to stay off**.
- The decoder recombines the active latents to reconstruct the original CLIP feature.

If this works well, some latents may correspond to interpretable recurring patterns such as pose, color, beak shape, background, or texture.


In [ ]:
@dataclass
class SAEConfig:
    d_in: int = 512
    d_sae: int = 4096
    l1_coeff: float = 2e-3
    lr: float = 2e-4
    n_epochs: int = 40
    batch_size: int = 256
    normalize_decoder: bool = True


class SparseAutoencoder(nn.Module):
    W_enc: Float[Tensor, "d_in d_sae"]
    W_dec: Float[Tensor, "d_sae d_in"]
    b_enc: Float[Tensor, "d_sae"]
    b_dec: Float[Tensor, "d_in"]

    def __init__(self, cfg: SAEConfig):
        super().__init__()
        self.cfg = cfg

        self.W_enc = nn.Parameter(t.empty(cfg.d_in, cfg.d_sae))
        self.W_dec = nn.Parameter(t.empty(cfg.d_sae, cfg.d_in))
        self.b_enc = nn.Parameter(t.zeros(cfg.d_sae))
        self.b_dec = nn.Parameter(t.zeros(cfg.d_in))

        nn.init.kaiming_uniform_(self.W_enc, nonlinearity="relu")
        nn.init.kaiming_uniform_(self.W_dec)
        self._normalise_decoder()

    def encode(self, x: Float[Tensor, "... d_in"]) -> Float[Tensor, "... d_sae"]:
        x_cent = x - self.b_dec
        pre = x_cent @ self.W_enc + self.b_enc
        return F.relu(pre)

    def decode(self, z: Float[Tensor, "... d_sae"]) -> Float[Tensor, "... d_in"]:
        return z @ self.W_dec + self.b_dec

    def forward(
        self, x: Float[Tensor, "batch d_in"]
    ) -> tuple[Float[Tensor, "batch d_sae"], Float[Tensor, "batch d_in"]]:
        z = self.encode(x)
        x_hat = self.decode(z)
        return z, x_hat

    @t.no_grad()
    def _normalise_decoder(self):
        norms = self.W_dec.norm(dim=-1, keepdim=True).clamp(min=1.0)
        self.W_dec.data /= norms

    def loss(
        self,
        x: Float[Tensor, "batch d_in"],
        z: Float[Tensor, "batch d_sae"],
        x_hat: Float[Tensor, "batch d_in"],
    ) -> tuple[Tensor, dict]:
        recon_loss = ((x - x_hat) ** 2).sum(dim=-1).mean()
        l1_loss = z.abs().sum(dim=-1).mean()
        total_loss = recon_loss + self.cfg.l1_coeff * l1_loss
        return total_loss, {
            "loss": float(total_loss.item()),
            "recon_loss": float(recon_loss.item()),
            "l1_loss": float(l1_loss.item()),
        }


In [ ]:
cfg_demo = SAEConfig(d_in=D_CLIP, d_sae=D_CLIP * 4)
sae_demo = SparseAutoencoder(cfg_demo).to(device)

x_dummy = t.randn(4, cfg_demo.d_in, device=device)
z_dummy, xhat_dummy = sae_demo(x_dummy)
loss_dummy, loss_dict_dummy = sae_demo.loss(x_dummy, z_dummy, xhat_dummy)

print("z shape:       ", tuple(z_dummy.shape))
print("x_hat shape:   ", tuple(xhat_dummy.shape))
print("loss dict keys:", list(loss_dict_dummy.keys()))
print("fraction active:", float((z_dummy > 0).float().mean()))

# Part 3: Training the SAE

Implement `train_sae`.

## Your task

For each epoch, your function should:

1. iterate over mini-batches from `train_feats`
2. run the SAE forward pass
3. compute the loss with `sae.loss(...)`
4. do backprop + optimizer step
5. optionally renormalize the decoder if `cfg.normalize_decoder` is `True`
6. record one log dictionary per epoch

Each epoch log should contain:
- `"loss"`
- `"recon_loss"`
- `"l1_loss"`
- `"dead_latent_pct"`

Here, `"dead_latent_pct"` means the percentage of latents that were **never active** during that epoch.

### Required function signature

```python
def train_sae(
    sae: SparseAutoencoder,
    train_feats: Float[Tensor, "n d_in"],
    cfg: SAEConfig,
) -> list[dict]:
```



### Implementation task: `train_sae`

In this section you will train the sparse autoencoder.

**What the training loop needs to do**
1. Put the model in training mode.
2. Create an Adam optimizer.
3. Iterate over minibatches from a `DataLoader`.
4. For each batch:
   - move the batch to `device`
   - run the SAE forward pass
   - compute the loss with `sae.loss(...)`
   - backpropagate and update parameters
5. Keep track of average loss values for the epoch.
6. Track what fraction of latents were **dead** in that epoch.

**What is a dead latent?**
A latent is "dead" in an epoch if it never has a positive activation on any training example in that epoch.

**Hints**
- `TensorDataset(train_feats)` gives a dataset whose batches look like `(x_batch,)`.
- `opt.zero_grad()`, `loss.backward()`, and `opt.step()` are the standard PyTorch update steps.
- The decoder rows are sometimes renormalized after each optimizer step. The config flag `cfg.normalize_decoder` tells you whether to do this.
- To compute the dead-latent percentage, you can keep a boolean tensor `ever_active` and update it with something like "did this latent fire anywhere in this batch?"


In [ ]:
def train_sae(
    sae: SparseAutoencoder,
    train_feats: Float[Tensor, "n d_in"],
    cfg: SAEConfig,
) -> list[dict]:
    """Train the SAE and return per-epoch logs."""
    # TODO: switch the model to training mode.
    sae.train()

    # TODO: create an Adam optimizer using cfg.lr.
    opt = t.optim.Adam(sae.parameters(), lr=cfg.lr)

    # TODO: create a DataLoader over train_feats with batch_size=cfg.batch_size
    # and shuffle=True.
    loader = DataLoader(TensorDataset(train_feats), batch_size=cfg.batch_size, shuffle=True)

    logs = []

    for _epoch in tqdm(range(cfg.n_epochs), desc="Training SAE"):
        epoch_totals = {"loss": 0.0, "recon_loss": 0.0, "l1_loss": 0.0}

        # TODO: create a boolean tensor called ever_active of shape (cfg.d_sae,)
        # on the correct device. It should start as all False.
        ever_active = t.zeros(cfg.d_sae, dtype=t.bool, device=device)

        # TODO: iterate over the DataLoader.
        # Each item from the loader is a 1-tuple, so unpack it as: for (x_batch,) in loader:
        for (x_batch,) in loader:
            # TODO: move x_batch to device.
            x_batch = x_batch.to(device)
            # TODO: zero gradients.
            opt.zero_grad()
            # TODO: run the SAE forward pass to get z and x_hat.
            z_batch, xhat_batch = sae(x_batch)
            # TODO: compute the loss and the loss dictionary.
            loss_batch, loss_dict_batch = sae.loss(x_batch, z_batch, xhat_batch)
            # TODO: backpropagate and take an optimizer step.
            loss_batch.backward()
            opt.step()  
            # TODO: if cfg.normalize_decoder is True, renormalize the decoder.
            if cfg.normalize_decoder:
                sae._normalise_decoder()    
            # TODO: add the current batch's scalar losses into epoch_totals.
            for key in epoch_totals.keys():
                epoch_totals[key] += loss_dict_batch[key] 
            # TODO: update ever_active so that a latent counts as active if it had
            # a positive activation for at least one example in this batch.
            ever_active |= (z_batch > 0).any(dim=0)

        # TODO: compute the mean of each tracked loss over all batches.
        # Store the result in a dictionary called epoch_log.
        num_batches = len(loader)
        epoch_log = {"loss": epoch_totals["loss"] / num_batches, "recon_loss": epoch_totals["recon_loss"] / num_batches, "l1_loss": epoch_totals["l1_loss"] / num_batches}
        # TODO: compute "dead_latent_pct" and add it to epoch_log.
        # This should be a percentage in [0, 100], not a fraction in [0, 1].
        epoch_log["dead_latent_pct"] = float((~ever_active).float().mean().item() * 100)

        # TODO: append epoch_log to logs.
        logs.append(epoch_log)

    return logs

In [ ]:
# Train the SAE on CLIP features
t.manual_seed(SEED)
cfg = SAEConfig(d_in=D_CLIP, d_sae=D_CLIP * 8, l1_coeff=2e-3, lr=2e-4, n_epochs=40, batch_size=256)
sae = SparseAutoencoder(cfg).to(device)

logs = train_sae(sae, train_feats, cfg)

In [ ]:
def plot_training_diagnostics(logs: list[dict]) -> None:
    epochs = np.arange(1, len(logs) + 1)
    loss  = [x["loss"]          for x in logs]
    recon = [x["recon_loss"]    for x in logs]
    l1    = [x["l1_loss"]       for x in logs]
    dead  = [x["dead_latent_pct"] for x in logs]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(epochs, loss,  label="total")
    axes[0].plot(epochs, recon, label="recon")
    axes[0].set_title("Total & reconstruction loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, l1, color="green", label="l1")
    axes[1].set_title("L1 loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    axes[2].plot(epochs, dead, color="orange")
    axes[2].set_title("Dead latent %")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("% dead")

    plt.tight_layout()
    plt.show()

In [ ]:
# Plot training diagnostics
plot_training_diagnostics(logs)

# Part 4: Latent retrieval & naming

Now that we have encoded all the examples into SAE latent space, we can ask the following:

1. Which latents activate often?
2. Which latents are more specific to one class than the other?
3. Which images activate a given latent most strongly?

This is often more interpretable than directly looking at raw CLIP dimensions.


In [ ]:
@t.no_grad()
def get_all_latents(
    sae: SparseAutoencoder,
    feats: Float[Tensor, "n d_in"],
    batch_size: int = 512,
) -> Float[Tensor, "n d_sae"]:
    sae.eval()
    chunks = []
    for start in range(0, len(feats), batch_size):
        x = feats[start : start + batch_size].to(device)
        z = sae.encode(x).cpu()
        chunks.append(z)
    return t.cat(chunks, dim=0)

train_latents = get_all_latents(sae, train_feats)
test_latents = get_all_latents(sae, test_feats)

print("Train latents shape:", tuple(train_latents.shape))
print("Test latents shape: ", tuple(test_latents.shape))
print("Average activation rate:", float((train_latents > 0).float().mean()))

In [ ]:
class0_name, class1_name = CLASS_NAMES[0], CLASS_NAMES[1]

class0_mask = train_labels == 0
class1_mask = train_labels == 1

class0_act_freq = (train_latents[class0_mask] > 0).float().mean(dim=0)
class1_act_freq = (train_latents[class1_mask] > 0).float().mean(dim=0)
specificity = class0_act_freq - class1_act_freq

abs_spec, sorted_idx = specificity.abs().sort(descending=True)

top_k = 15
table = Table(title=f"Top {top_k} class-specific latents")
table.add_column("latent")
table.add_column(f"{class0_name} freq")
table.add_column(f"{class1_name} freq")
table.add_column("specificity")
table.add_column("prefers")

for latent_idx in sorted_idx[:top_k].tolist():
    f0 = float(class0_act_freq[latent_idx])
    f1 = float(class1_act_freq[latent_idx])
    sp = float(specificity[latent_idx])
    table.add_row(
        str(latent_idx),
        f"{f0:.3f}",
        f"{f1:.3f}",
        f"{sp:+.3f}",
        class0_name if sp > 0 else class1_name,
    )

rprint(table)

top_class0_latent = int(specificity.argmax().item())
top_class1_latent = int(specificity.argmin().item())
print(f"Top {class0_name}-preferring latent:", top_class0_latent)
print(f"Top {class1_name}-preferring latent:", top_class1_latent)


# Retrieving the top-activating examples

A standard SAE interpretability trick is:

- choose one latent,
- find the images where it is largest,
- inspect those images for a common pattern.

## Your task

Implement `get_top_activating_examples`.

### Required behavior

Given a latent index and the matrix of latent activations:
1. extract the activation vector for that latent across all examples
2. find the top-`k` activations
3. return both:
   - the indices of those examples
   - the activation values themselves

### Required function signature

```python
def get_top_activating_examples(
    latent_idx: int,
    latents: Float[Tensor, "n d_sae"],
    k: int = 10,
) -> tuple[Int[Tensor, "k"], Float[Tensor, "k"]]:
```



### Implementation task: `get_top_activating_examples`

Given a latent index and a matrix of latent activations, return the indices of the top-`k` examples for that latent.

**Hints**
- `latents[:, latent_idx]` selects the activations of one latent across all examples.
- In PyTorch, `topk(k)` returns both the values and the indices.
- Be careful about the return order: the function should return `(top_idx, top_vals)`, not `(top_vals, top_idx)`.


In [ ]:
def get_top_activating_examples(
    latent_idx: int,
    latents: Float[Tensor, "n d_sae"],
    k: int = 10,
) -> tuple[Int[Tensor, "k"], Float[Tensor, "k"]]:
    """Return indices and values of the top-k examples for one latent."""
    # TODO: extract the activation vector for the chosen latent.
    acts = latents[:, latent_idx]

    # TODO: use topk to get the top activation values and their indices.
    top_vals, top_idx = acts.topk(k)

    return top_idx, top_vals

In [ ]:
def show_top_activating_images(
    latent_idx: int,
    latents: Float[Tensor, "n d_sae"],
    dataset,
    labels: Int[Tensor, "n"],
    k: int = 10,
) -> None:
    idx, vals = get_top_activating_examples(latent_idx, latents, k=k)

    n_cols = min(5, k)
    n_rows = int(np.ceil(k / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
    axes = np.array(axes).reshape(-1)

    for rank, (img_idx, act_val) in enumerate(zip(idx.tolist(), vals.tolist())):
        class_name = CLASS_NAMES[int(labels[img_idx].item())]
        axes[rank].imshow(dataset[img_idx]["image"].convert("RGB"))
        axes[rank].set_title(f"{class_name}\nact={act_val:.2f}", fontsize=9)
        axes[rank].axis("off")

    for ax in axes[k:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
show_top_activating_images(top_class0_latent, train_latents, train_ds, train_labels, k=8)
show_top_activating_images(top_class1_latent, train_latents, train_ds, train_labels, k=8)

## Auto-naming latents with CLIP text

One of the most satisfying tools in SAE interpretability is being able to **name** what a latent detects, not just by looking at images, but by finding which text descriptions best match the latent's top-activating images.

Because CLIP was trained to align images and text in a shared embedding space, we can do this for free: the mean CLIP feature of a latent's top-activating images can be directly compared to text embeddings using cosine similarity.

### Your task

Fill in `CANDIDATE_TEXTS` below with at least 10 text descriptions that you think might describe visual concepts relevant to puffins and penguins. Think about:
- distinctive physical features (beak shape, colouring, crests)
- habitat and background (cliffs, ice, ocean)
- behaviour (swimming, flying, huddling)
- neutral descriptions that could apply to both
```python
CANDIDATE_TEXTS = [
    # YOUR DESCRIPTIONS HERE
]
```

It might help to look at the top activating images for each class to get some ideas. Once you have filled in your candidate texts, run the cells below to see which descriptions best match the most class-specific latents, and which training images are closest to each description.

In [ ]:
# TODO: fill in candidate texts with a minimum of 10 examples
CANDIDATE_TEXTS = [
    # TODO: Puffin-associated texts
    "a bird with a bright orange beak",
    "a puffin standing on a rocky cliff",
    "a small seabird with a colorful beak",
    "a puffin floating in the ocean",
    "a puffin with wings spread on a rock",

    # TODO: Penguin-associated texts
    "a penguin standing upright on rocky ground",
    "a penguin colony gathered together",
    "a black and white bird in a cold environment",
    "a penguin walking near the shore",
    "a penguin diving or swimming in the water",

    # TODO: Neutral texts
    "a seabird near the ocean",
    "a bird standing on rocks",
    "a bird in a coastal environment",
    "a black and white bird outdoors",
    "a bird near water"
]

In [ ]:
@t.no_grad()
def get_text_embeddings(texts: list[str]) -> Float[Tensor, "n_texts d_clip"]:
    inputs = clip_processor(text=texts, return_tensors="pt", padding=True).to(device)
    outputs = clip_model.text_model(**inputs)
    text_feats = clip_model.text_projection(outputs.pooler_output)
    return F.normalize(text_feats, dim=-1).cpu()

In [ ]:
def show_autoname_with_images(
    latent_idx: int,
    latents: Float[Tensor, "n d_sae"],
    feats: Float[Tensor, "n d_in"],
    labels: Int[Tensor, "n"],
    dataset,
    text_embeddings: Float[Tensor, "n_texts d_in"],
    candidate_texts: list[str],
    k: int = 5,
) -> None:
    """
    For each top-matching text description, find and display the training image
    whose CLIP feature is closest to that text embedding.
    """
    # Get top-k text matches for this latent
    top_idx = latents[:, latent_idx].topk(20).indices
    mean_feat = F.normalize(feats[top_idx].mean(0), dim=0)
    sims = (text_embeddings @ mean_feat).numpy()
    top_text_idx = sims.argsort()[::-1][:k]

    fig, axes = plt.subplots(1, k, figsize=(3 * k, 4))
    class_names = ["Penguin", "Puffin"]

    for col, text_idx in enumerate(top_text_idx):
        # Find the training image closest to this text embedding
        text_emb = text_embeddings[text_idx]
        img_sims = (feats @ text_emb).numpy()
        best_img_idx = int(img_sims.argmax())

        img = dataset[best_img_idx]["image"].convert("RGB")
        cls = class_names[labels[best_img_idx].item()]

        axes[col].imshow(img)
        # Wrap long text for the title
        words = candidate_texts[text_idx].split()
        wrapped = "\n".join([" ".join(words[i:i+4]) for i in range(0, len(words), 4)])
        axes[col].set_title(f"{wrapped}\nsim={sims[text_idx]:.3f} ({cls})", fontsize=8)
        axes[col].axis("off")

    fig.suptitle(f"Latent {latent_idx} — top matching descriptions + nearest images", fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
# Run this once to get text embeddings for your candidate texts
text_embeddings = get_text_embeddings(CANDIDATE_TEXTS)

# Visualise the top matching descriptions and nearest images for each class-specific latent
show_autoname_with_images(top_class0_latent, train_latents, train_feats, train_labels, train_ds, text_embeddings, CANDIDATE_TEXTS)
show_autoname_with_images(top_class1_latent, train_latents, train_feats, train_labels, train_ds, text_embeddings, CANDIDATE_TEXTS)